# Harmonic regression and complex SVD of multi-tissue rhythms

**Dataset:** young and aged mouse tissues sampled every 3 hours.

**Source paper:** Wang et al.,
[*Redox rhythms promote fitness by modulating ageing-dependent reprogramming*](https://www.nature.com/articles/s42255-026-01515-x),
Nature Metabolism (2026). The PDF is included as
`s42255-026-01515-x.pdf`.

In this workshop, we will:

1. fit a 24-hour harmonic model to every gene;
2. apply Benjamini-Hochberg correction independently per tissue and age;
3. count rhythmic genes using BH and amplitude thresholds;
4. compare young and aged tissues;
5. make cumulative amplitude-threshold plots; and
6. perform complex SVD with several gene lists.

The notebook uses compressed CSV mirrors of the FPKM columns. The
original `FPKM_JTK.xlsx` workbook is included in the `data` folder.

## 1. Imports and editable parameters

The main analysis parameters are in the next cell.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy.stats import beta

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = Path("data/fpkm_csv")
RESULT_DIR = Path("results")
RESULT_DIR.mkdir(exist_ok=True)

ALL_TISSUES = ["TA", "AA", "AR", "TH", "LH", "RH",
               "SM", "LV", "BF", "SP", "WAT", "KD"]
TISSUES = ALL_TISSUES

PERIOD = 24
BH_THRESHOLD = 0.05
AMPLITUDE_THRESHOLD = 0.5
TOP_PAN_GENES = 200

TISSUE_NAMES = {
    "TA": "Thoracic aorta", "AA": "Abdominal aorta",
    "AR": "Aortic arch", "TH": "Tip of heart",
    "LH": "Left heart", "RH": "Right heart",
    "SM": "Skeletal muscle", "LV": "Liver",
    "BF": "Brown fat", "SP": "Spleen",
    "WAT": "White adipose tissue", "KD": "Kidney",
}
TISSUE_COLORS = dict(
    zip(ALL_TISSUES, plt.cm.tab20(np.linspace(0, 1, len(ALL_TISSUES))))
)

CLOCK_GENES = [
    "Arntl", "Clock", "Npas2", "Per1", "Per2", "Per3",
    "Cry1", "Cry2", "Nr1d1", "Nr1d2", "Ciart", "Rora",
    "Rorc", "Dbp", "Nfil3", "Tef", "Hlf",
]

print("Tissues:", TISSUES)
print("Rhythmicity: BH <", BH_THRESHOLD,
      "and log2 amplitude >", AMPLITUDE_THRESHOLD)

## 2. Load FPKM expression and sample metadata

Sample names encode tissue, age (`Y` or `A`), RNA-seq sample and ZT.
Six mice were collected per tissue, age, and time point. Three mice
were pooled before RNA sequencing, producing two RNA-seq samples per
tissue, age, and time point. Values are transformed as
`log2(FPKM + 1)` before model fitting.

In [ ]:
SAMPLE_PATTERN = re.compile(
    r"^(?P<tissue>[A-Z]+)(?P<age>[YA])(?P<rna_seq_sample>\d+)"
    r"ZT(?P<time>\d+)_FPKM$"
)

def load_tissue(tissue):
    table = pd.read_csv(DATA_DIR / f"{tissue}_FPKM.csv.gz")
    sample_columns = [c for c in table.columns if c.endswith("_FPKM")]
    metadata = []
    for column in sample_columns:
        match = SAMPLE_PATTERN.match(column)
        if match is None:
            raise ValueError(f"Cannot parse sample column: {column}")
        row = match.groupdict()
        row["sample"] = column
        row["age"] = {"Y": "Young", "A": "Aged"}[row["age"]]
        row["time"] = float(row["time"])
        row["rna_seq_sample"] = int(row["rna_seq_sample"])
        metadata.append(row)

    expression = table.set_index("gene_id")[sample_columns]
    expression = expression.apply(pd.to_numeric, errors="coerce")
    expression = np.log2(expression.fillna(0.0) + 1.0)
    return expression, pd.DataFrame(metadata)

example_expression, example_metadata = load_tissue(TISSUES[0])
display(example_metadata.head())
print("Example expression matrix:", example_expression.shape)

## 3. Harmonic regression

We fit

$$
x(t) = \mu
+ a \cos\left(\frac{2\pi t}{24}\right)
+ b \sin\left(\frac{2\pi t}{24}\right)
+ \epsilon
$$

The peak-to-trough harmonic amplitude is
`2*sqrt(a**2 + b**2)`, and phase is derived from `atan2(b, a)`.
Rhythmicity is tested from the model R² using its null beta
distribution, followed by Benjamini-Hochberg correction across genes.

In [ ]:
def benjamini_hochberg(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    adjusted = np.full(pvalues.shape, np.nan)
    valid = np.isfinite(pvalues)
    p = pvalues[valid]
    order = np.argsort(p)
    ranked = p[order]
    corrected = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    corrected = np.minimum.accumulate(corrected[::-1])[::-1]
    restored = np.empty_like(corrected)
    restored[order] = np.minimum(corrected, 1.0)
    adjusted[valid] = restored
    return adjusted


def fit_harmonic_matrix(expression, times, period=24):
    values = np.asarray(expression, dtype=float)
    times = np.asarray(times, dtype=float)
    design = np.column_stack([
        np.ones(len(times)),
        np.cos(2 * np.pi * times / period),
        np.sin(2 * np.pi * times / period),
    ])

    coefficients, _, _, _ = np.linalg.lstsq(
        design, values.T, rcond=None
    )
    fitted = design @ coefficients
    residual = values.T - fitted
    sse = np.sum(residual ** 2, axis=0)
    centered = values - values.mean(axis=1, keepdims=True)
    tss = np.sum(centered ** 2, axis=1)
    r2 = np.where(tss > 0, 1 - sse / tss, 0.0)
    r2 = np.clip(r2, 0.0, 1.0)

    n = values.shape[1]
    pvalues = beta.sf(r2, 1.0, (n - 3) / 2.0)
    mu, a, b = coefficients

    result = pd.DataFrame({
        "gene": expression.index.astype(str),
        "mean": values.mean(axis=1),
        "amp": 2 * np.sqrt(a ** 2 + b ** 2),
        "phase": (period / (2 * np.pi) * np.arctan2(b, a)) % period,
        "pval": pvalues,
        "a": a,
        "b": b,
        "R2": r2,
        "n_samples": n,
    })
    result["BH"] = benjamini_hochberg(result["pval"])
    return result


def fit_all_tissues(tissues):
    fitted = []
    for tissue in tissues:
        print("Fitting", tissue)
        expression, metadata = load_tissue(tissue)
        for age in ["Young", "Aged"]:
            selected = metadata["age"].eq(age).to_numpy()
            fit = fit_harmonic_matrix(
                expression.loc[:, selected],
                metadata.loc[selected, "time"],
                period=PERIOD,
            )
            fit["tissue"] = tissue
            fit["age"] = age
            fitted.append(fit)
    return pd.concat(fitted, ignore_index=True)


fits = fit_all_tissues(TISSUES)
fits["rhythmic"] = (
    (fits["BH"] < BH_THRESHOLD)
    & (fits["amp"] > AMPLITUDE_THRESHOLD)
)
fits.to_csv(
    RESULT_DIR / "genomewide_harmonic_results.csv.gz",
    index=False, compression="gzip"
)
display(fits.head())

### Exercise 1

Inspect one clock gene in one tissue. Plot the observed points and
fitted harmonic curves in young and aged mice. Try `Arntl`, `Dbp`,
`Per2`, or `Nr1d1`.

In [ ]:
def plot_gene_fit(gene="Arntl", tissue="LH"):
    expression, metadata = load_tissue(tissue)
    fig, ax = plt.subplots(figsize=(7, 4))
    dense_time = np.linspace(0, 24, 300)
    for age, color in [("Young", "#377eb8"), ("Aged", "#e41a1c")]:
        selected = metadata["age"].eq(age).to_numpy()
        observed = expression.loc[gene, selected].to_numpy()
        time = metadata.loc[selected, "time"].to_numpy()
        fit = fits.query(
            "gene == @gene and tissue == @tissue and age == @age"
        ).iloc[0]
        predicted = (
            fit["mean"]
            + fit["a"] * np.cos(2 * np.pi * dense_time / PERIOD)
            + fit["b"] * np.sin(2 * np.pi * dense_time / PERIOD)
        )
        ax.scatter(time, observed, color=color, alpha=0.7, label=age)
        ax.plot(dense_time, predicted, color=color)
    ax.set(
        title=f"{gene} in {TISSUE_NAMES[tissue]}",
        xlabel="Zeitgeber time (h)",
        ylabel="log2(FPKM + 1)",
        xticks=np.arange(0, 25, 3),
    )
    ax.legend()
    plt.show()

plot_gene_fit("Arntl", "LH" if "LH" in TISSUES else TISSUES[0])

## 4. Number of rhythmic genes: young versus aged

A gene is rhythmic when both criteria are satisfied:

- BH-adjusted P value below `BH_THRESHOLD`;
- harmonic amplitude above `AMPLITUDE_THRESHOLD`.

In [ ]:
rhythmic_counts = (
    fits.groupby(["tissue", "age"], as_index=False)["rhythmic"]
    .sum()
    .rename(columns={"rhythmic": "rhythmic_genes"})
)
count_table = (
    rhythmic_counts.pivot(
        index="tissue", columns="age", values="rhythmic_genes"
    )
    .reindex(TISSUES)
    .reset_index()
)
count_table["aged_minus_young"] = (
    count_table["Aged"] - count_table["Young"]
)
count_table["aged_over_young"] = (
    count_table["Aged"] / count_table["Young"]
)
count_table.to_csv(
    RESULT_DIR / "rhythmic_gene_counts_young_vs_aged.csv", index=False
)
display(count_table)

maximum = count_table[["Young", "Aged"]].to_numpy().max() * 1.08
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, maximum], [0, maximum], "--", color="0.55")
point_colors = [TISSUE_COLORS[tissue] for tissue in count_table["tissue"]]
ax.scatter(
    count_table["Young"], count_table["Aged"],
    c=point_colors, s=70, edgecolor="white", linewidth=0.7
)
for row in count_table.itertuples():
    ax.annotate(
        row.tissue, (row.Young, row.Aged),
        xytext=(4, 4), textcoords="offset points"
    )
ax.set(
    xlim=(0, maximum), ylim=(0, maximum),
    aspect="equal",
    xlabel="Rhythmic genes in young mice",
    ylabel="Rhythmic genes in aged mice",
    title="Young versus aged rhythmic-gene counts",
)
plt.show()

### Exercise 2

Change the amplitude threshold to 0.25, 0.75 or 1.0. Which tissues are
robust to the choice? Which tissues remain far below the identity line?

## 5. Cumulative rhythmic-gene count versus amplitude

BH remains fixed while the minimum amplitude is progressively raised.
Each curve answers: *how many genes have BH below the cutoff and
amplitude greater than this value?*

In [ ]:
amplitude_grid = np.round(np.arange(0, 3.01, 0.1), 2)
cumulative_rows = []
for threshold in amplitude_grid:
    selected = (
        (fits["BH"] < BH_THRESHOLD)
        & (fits["amp"] > threshold)
    )
    counts = (
        fits.assign(selected=selected)
        .groupby(["tissue", "age"], as_index=False)["selected"]
        .sum()
    )
    counts["amplitude_threshold"] = threshold
    cumulative_rows.append(counts)

cumulative = pd.concat(cumulative_rows, ignore_index=True)
cumulative = cumulative.rename(
    columns={"selected": "rhythmic_genes"}
)
cumulative.to_csv(
    RESULT_DIR / "rhythmic_genes_by_amplitude_threshold.csv",
    index=False
)

fig, axes = plt.subplots(
    1, 2, figsize=(13, 5), sharex=True, sharey=True
)
for ax, age in zip(axes, ["Young", "Aged"]):
    age_data = cumulative.query("age == @age")
    for tissue in TISSUES:
        subset = age_data.query("tissue == @tissue")
        ax.plot(
            subset["amplitude_threshold"],
            subset["rhythmic_genes"],
            label=tissue, linewidth=1.8
        )
    ax.axvline(
        AMPLITUDE_THRESHOLD, linestyle="--", color="0.5"
    )
    ax.set(
        title=age, xlabel="Minimum log2 amplitude",
        ylabel="Number of rhythmic genes"
    )
axes[1].legend(
    title="Tissue", bbox_to_anchor=(1.02, 1), loc="upper left"
)
fig.suptitle(f"Cumulative counts at BH < {BH_THRESHOLD}")
fig.tight_layout()
plt.show()

## 6. Select gene lists for complex SVD

The pan-rhythmic ranking prioritizes genes rhythmic in the same tissue
in both ages, then genes rhythmic in either age, then total datasets.
Core clock genes are excluded from the pan-rhythmic list.

In [ ]:
rhythmic_matrix = (
    fits.pivot_table(
        index="gene", columns=["age", "tissue"],
        values="rhythmic", aggfunc="first", fill_value=False
    )
    .astype(bool)
)
amplitude_matrix = fits.pivot_table(
    index="gene", columns=["age", "tissue"],
    values="amp", aggfunc="first"
)

available_tissues = [
    tissue for tissue in TISSUES
    if ("Young", tissue) in rhythmic_matrix.columns
    and ("Aged", tissue) in rhythmic_matrix.columns
]
young = rhythmic_matrix.loc[
    :, [("Young", tissue) for tissue in available_tissues]
].copy()
aged = rhythmic_matrix.loc[
    :, [("Aged", tissue) for tissue in available_tissues]
].copy()
young.columns = available_tissues
aged.columns = available_tissues

ranking = pd.DataFrame(index=rhythmic_matrix.index)
ranking["n_tissues_both"] = (young & aged).sum(axis=1)
ranking["n_tissues_any"] = (young | aged).sum(axis=1)
ranking["n_datasets"] = rhythmic_matrix.sum(axis=1)
ranking["mean_amplitude"] = amplitude_matrix.mean(axis=1)
ranking = (
    ranking.loc[
        ~ranking.index.str.lower().isin(
            {gene.lower() for gene in CLOCK_GENES}
        )
    ]
    .sort_values(
        ["n_tissues_both", "n_tissues_any",
         "n_datasets", "mean_amplitude"],
        ascending=False,
    )
)
ranking.to_csv(RESULT_DIR / "pan_rhythmic_gene_ranking.csv")
PAN_RHYTHMIC_GENES = ranking.head(TOP_PAN_GENES).index.tolist()

# Edit or replace this list with genes of interest.
CUSTOM_GENES = ["Arntl", "Per2", "Cry1", "Nr1d1", "Dbp"]

GENE_LISTS = {
    "core_clock": CLOCK_GENES,
    "top_pan_rhythmic": PAN_RHYTHMIC_GENES,
    "custom": CUSTOM_GENES,
}
{name: len(genes) for name, genes in GENE_LISTS.items()}

## 7. Complex SVD

For each gene and tissue-age dataset, define the complex harmonic
coefficient

$$
z = a + i b
$$

Coefficients that do not pass the selected BH and amplitude thresholds
are set to zero. SVD is then applied directly to this complex matrix.

In [ ]:
def build_complex_matrix(genes):
    datasets = [
        (age, tissue)
        for tissue in TISSUES
        for age in ["Young", "Aged"]
    ]
    matrix = np.zeros(
        (len(genes), len(datasets)), dtype=np.complex128
    )
    for column, (age, tissue) in enumerate(datasets):
        subset = (
            fits.query("age == @age and tissue == @tissue")
            .set_index("gene")
        )
        index = subset.index.intersection(genes)
        positions = {gene: i for i, gene in enumerate(genes)}
        valid = (
            (subset.loc[index, "BH"] < BH_THRESHOLD)
            & (subset.loc[index, "amp"] > AMPLITUDE_THRESHOLD)
        )
        selected = index[valid]
        for gene in selected:
            matrix[positions[gene], column] = (
                subset.at[gene, "a"] + 1j * subset.at[gene, "b"]
            )
    return matrix, datasets


def run_csvd(genes, mode=1):
    genes = list(dict.fromkeys(genes))
    matrix, datasets = build_complex_matrix(genes)
    keep_genes = np.any(np.abs(matrix) > 0, axis=1)
    keep_datasets = np.any(np.abs(matrix) > 0, axis=0)
    matrix = matrix[np.ix_(keep_genes, keep_datasets)]
    kept_genes = np.asarray(genes)[keep_genes]
    kept_datasets = np.asarray(datasets, dtype=object)[keep_datasets]

    if min(matrix.shape) < 2:
        raise ValueError("Too few non-zero genes or datasets for cSVD")

    u, singular_values, vh = np.linalg.svd(
        matrix, full_matrices=False
    )
    v = vh.conj().T

    # Rotate each mode to make tissue-space orientation interpretable.
    for k in range(len(singular_values)):
        total = v[:, k].sum()
        maximum = np.abs(v[:, k]).max()
        if np.abs(total) == 0 or maximum == 0:
            continue
        rotation = np.conj(total) / np.abs(total)
        u[:, k] *= rotation * maximum * singular_values[k]
        v[:, k] = np.conj(v[:, k] * rotation / maximum)

    component = mode - 1
    variance = (
        singular_values[component] ** 2
        / np.sum(singular_values ** 2) * 100
    )
    return {
        "matrix": matrix, "genes": kept_genes,
        "datasets": kept_datasets, "u": u, "v": v,
        "singular_values": singular_values,
        "mode": component, "variance": variance,
    }


def plot_csvd(result, title):
    component = result["mode"]
    gene_scores = result["u"][:, component]
    tissue_scores = result["v"][:, component]
    datasets = result["datasets"]

    fig = plt.figure(figsize=(13, 5.5))
    gene_ax = fig.add_subplot(1, 2, 1, projection="polar")
    tissue_ax = fig.add_subplot(1, 2, 2, projection="polar")

    gene_ax.scatter(
        np.angle(gene_scores) % (2 * np.pi),
        np.abs(gene_scores), s=16, alpha=0.7
    )
    label_count = min(25, len(gene_scores))
    for index in np.argsort(np.abs(gene_scores))[-label_count:]:
        gene_ax.text(
            np.angle(gene_scores[index]) % (2 * np.pi),
            np.abs(gene_scores[index]),
            result["genes"][index], fontsize=7
        )
    gene_ax.set_title("Gene space")

    markers = {"Young": "o", "Aged": "^"}
    for score, (age, tissue) in zip(tissue_scores, datasets):
        tissue_ax.scatter(
            np.angle(score) % (2 * np.pi), np.abs(score),
            marker=markers[age], color=TISSUE_COLORS[tissue],
            s=60, edgecolor="white", linewidth=0.6
        )
    tissue_ax.set_title("Tissue-age space")

    for ax in [gene_ax, tissue_ax]:
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)
        ax.set_xticks(np.arange(0, 24, 3) / 24 * 2 * np.pi)
        ax.set_xticklabels([
            "ZT0/24\nLights on", "ZT3", "ZT6", "ZT9",
            "ZT12\nLights off", "ZT15", "ZT18", "ZT21",
        ])

    used_tissues = list(dict.fromkeys(tissue for _, tissue in datasets))
    tissue_handles = [
        Line2D(
            [0], [0], marker="o", linestyle="none",
            markerfacecolor=TISSUE_COLORS[tissue],
            markeredgecolor="white", markersize=8, label=tissue
        )
        for tissue in used_tissues
    ]
    age_handles = [
        Line2D(
            [0], [0], marker=markers[age], linestyle="none",
            color="0.25", markersize=8, label=age
        )
        for age in ["Young", "Aged"]
    ]
    tissue_legend = tissue_ax.legend(
        handles=tissue_handles, title="Tissue",
        bbox_to_anchor=(1.18, 1.05), loc="upper left",
        frameon=False, ncol=1
    )
    tissue_ax.add_artist(tissue_legend)
    tissue_ax.legend(
        handles=age_handles, title="Age",
        bbox_to_anchor=(1.18, 0.25), loc="upper left",
        frameon=False
    )

    fig.suptitle(
        f"{title}: mode {component + 1} "
        f"({result['variance']:.1f}% variance)"
    )
    fig.subplots_adjust(right=0.78, wspace=0.35)
    plt.show()


clock_csvd = run_csvd(GENE_LISTS["core_clock"])
plot_csvd(clock_csvd, "Core clock genes")

### Exercise 3

Run the same cSVD with:

1. `GENE_LISTS["top_pan_rhythmic"]`;
2. a custom pathway or gene list;
3. mode 2 instead of mode 1.

Compare the tissue-space phase and magnitude patterns. Remember that a
cSVD magnitude is a mode scaling factor, not a direct expression
amplitude.

In [ ]:
pan_csvd = run_csvd(GENE_LISTS["top_pan_rhythmic"])
plot_csvd(pan_csvd, f"Top {TOP_PAN_GENES} pan-rhythmic genes")

## 8. Optional heart-focused classification

For each heart region, classify genes as:

- **common:** rhythmic in young and aged;
- **lost:** rhythmic only in young;
- **gained:** rhythmic only in aged.

In [ ]:
heart_tissues = [t for t in ["TH", "LH", "RH"] if t in TISSUES]
if heart_tissues:
    heart = (
        fits.query("tissue in @heart_tissues")
        .pivot_table(
            index=["gene", "tissue"], columns="age",
            values="rhythmic", aggfunc="first", fill_value=False
        )
        .reset_index()
    )
    heart["status"] = np.select(
        [
            heart["Young"] & heart["Aged"],
            heart["Young"] & ~heart["Aged"],
            ~heart["Young"] & heart["Aged"],
        ],
        ["Common", "Lost", "Gained"],
        default="Neither",
    )
    heart_summary = (
        heart.query("status != 'Neither'")
        .groupby(["tissue", "status"])
        .size()
        .unstack(fill_value=0)
        .reindex(heart_tissues)
    )
    display(heart_summary)
    heart_summary.plot(
        kind="bar", stacked=True, figsize=(7, 4),
        color={"Common": "#4C78A8", "Lost": "#E45756",
               "Gained": "#59A14F"}
    )
    plt.ylabel("Genes")
    plt.title("Heart-region rhythmic reprogramming")
    plt.xticks(rotation=0)
    plt.show()

## Conclusions and discussion

Suggested questions:

- Does ageing reduce rhythmic-gene number uniformly across tissues?
- Are retained rhythms weaker, phase-shifted, or simply fewer?
- Does the core-clock cSVD tell the same story as the pan-rhythmic cSVD?
- How sensitive are conclusions to BH and amplitude thresholds?
- What are the limitations of using FPKM and only two deposited profiles
  per time point?